In [ ]:
import os
import sys
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
import json
import time
from datetime import datetime
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter, SpanExportResult, ConsoleSpanExporter
from opentelemetry.trace import Status, StatusCode

In [ ]:
class TraceJsonExporter(SpanExporter): # 要求符合 SpanExporter 規格
    def __init__(self, base_name = "trace"):
        self.base_name = base_name

    def export(self, spans):
        date = datetime.now().strftime("%Y%m%d")
        file_name = f"{self.base_name}_{date}.jsonl"

        with open(file_name, "a", encoding="utf-8") as f:
            for span in spans :
                trace = {
                    "name": span.name, # 後面呼叫 with tracer.start_as_current_span("llm_generation_step") as span 時就會填入 span name
                    "trace_id": format(span.get_span_context().trace_id, '032x'), # 在 opentelemetry 運算中 trace id 原為 128 bits (16 bytes), 轉成 32 字元
                    "span_id": format(span.get_span_context().span_id, '032x'),
                    "parent_span_id": format(span.parent.span_id, '032x') if span.parent else None,
                    "start_time": span.start_time,
                    "end_time": span.end_time,
                    "duration_ns": span.end_time - span.start_time,
                    "status": span.status.status_code.name, # span.status.status_code 會是狀態碼, .name 直接輸出狀態 (OK/ERROR)
                    "attribution": dict(span.attributes), # dict() 轉成 dictionary 格式以符合 json 儲存需求
                    "events": [
                        {
                        "event_name": event.name,
                        "timestamp": event.timestamp,
                        "attributes": dict(event.attributes) if event.attributes else {}
                        } for event in span.events
                    ]
                }
                f.write(json.dumps(trace, ensure_ascii=False)+"\n")
                    # .dump 意為 dump string, 將 pythin dict 轉為 json 儲存用的字串
                    # json 預設會將非 ASCII 字元(中文字)轉成代碼, ensure_ascii=False 強制保留原字串
        return SpanExportResult.SUCCESS # 告知系統此輪成功結束

    def shutdown(self):
        pass

In [ ]:
# Trace 格式
# Trace (由 Trace ID 標識：例如 5b3c8...)
# └── Span: "llm_generation_step" (由 Span ID 標識：例如 a1b2c...)
#     ├── Context (身份證)
#     │   ├── Trace ID (128-bit) -> 轉為 32 位 16 進制字串
#     │   └── Span ID (64-bit)   -> 轉為 16 位 16 進制字串
#     ├── Attributes (屬性：詳細數據)
#     │   ├── llm.model: "llama-3.3-70b-versatile"
#     │   ├── llm.temperature: 0.7
#     │   ├── input.value: "使用者問的問題"
#     │   ├── output.value: "AI 回答的內容"
#     │   └── usage.total_tokens: 512
#     ├── Events (事件：時間點標記)
#     │   └── "Sent request to API" (含時間戳記)
#     └── Status (狀態)
#         └── StatusCode: OK 或 ERROR

# ex.
# {
#   "name": "llm_generation_step",
#   "trace_id": "5b3c8f92a1d4e5f67890abcdef123456",
#   "span_id": "a1b2c3d4e5f6g7h8",
#   "start_time": 1713685200000000000,
#   "end_time": 1713685202500000000,
#   "status": "OK",
#   "attributes": {
#     "llm.model": "llama-3.3-70b-versatile",
#     "llm.temperature": 0.7,
#     "input.value": "你好，請介紹一下什麼是 Trace？",
#     "output.value": "你好！Trace（追蹤）在軟體工程中是指紀錄程式執行的路徑...",
#     "usage.prompt_tokens": 25,
#     "usage.completion_tokens": 150,
#     "usage.total_tokens": 175
#   },
#   "events": [
#     {
#       "name": "Sent request to API",
#       "timestamp": 1713685200500000000
#     }
#   ]
# }
#

In [ ]:
# Event 物件結構
# Event(
#     name="Sent request to API",
#     timestamp=1713685200500000000,
#     attributes={
#         "custom_key": "custom_value"
#     },
#     # 還有一些底層的內部引用...
# )

In [ ]:
class TraceTerminalExporter(SpanExporter):
    def export(self, spans):
        for span in spans:
            current_time = datetime.now().strftime("%Y%m%d, %H:%M:%S")
            model = span.attributes["model"]
            temperature = span.attributes["temperature"]
            duration_ns = span.end_time - span.start_time
            start_time = (datetime.fromtimestamp(span.start_time/1E9)).strftime("%H:%M:%S.%f")[:-2]
            end_time = (datetime.fromtimestamp(span.end_time/1E9)).strftime("%H:%M:%S.%f")[:-2]
            print(f"[{current_time}] {model} | span_name: {span.name} | temperature: {temperature} | start/end_time: {start_time}/{end_time} | duration_ns: {(duration_ns/1E9):.4f} s")
        return SpanExportResult.SUCCESS

    def shutdown(self):
        pass

In [ ]:
# trace/span 結構
# {
#     "name": "llm_generation",
#     "context": {
#         "trace_id": "0xbc0c6d9abe661b2fba6e5f8760ca42ff",
#         "span_id": "0xbc6def111e29704b",
#         "trace_state": "[]"
#     },
#     "kind": "SpanKind.INTERNAL",
#     "parent_id": null,
#     "start_time": "2026-04-22T01:38:40.126720Z",
#     "end_time": "2026-04-22T01:38:41.146903Z",
#     "status": {
#         "status_code": "OK"
#     },
#     "attributes": {
#         "model": "llama-3.3-70b-versatile",
#         "temperature": 1.0,
#         "max_tokens": 512,
#         "presence_penalty": 1.1,
#         "user_prompt": "\u4f60\u597d",
#         "assistant_response": "\u4f60\u597d\u3002\u5f88\u9ad8\u5174\u89c1\u5230\u4f60\uff0c\u6211\u53ef\u4ee5\u5982\u4f55\u5e2e\u52a9\u4f60\u5417\uff1f",
#         "prompt_token": 53,
#         "completion_token": 17,
#         "total_tokens": 70
#     },
#     "events": [
#         {
#             "name": "call_LLM_by_API",
#             "timestamp": "2026-04-22T01:38:40.126766Z",
#             "attributes": {}
#         }
#     ],
#     "links": [],
#     "resource": {
#         "attributes": {
#             "service.name": "LLM_test"
#         },
#         "schema_url": ""
#     }
# }

In [ ]:
resources = Resource(attributes={
    "service.name": "LLM_test",
})
provider = TracerProvider(resource=resources) # TracerProvider() 初始化本地追蹤器

file_exporter = TraceJsonExporter() # 定義輸出規格及目的地
file_processor = SimpleSpanProcessor(file_exporter) # 建立傳輸管道
provider.add_span_processor(file_processor)

terminal_exporter = TraceTerminalExporter()
terminal_processor = SimpleSpanProcessor(terminal_exporter)
provider.add_span_processor(terminal_processor)

tracer = provider.get_tracer(__name__)

In [ ]:
load_dotenv() # 讀取根目錄檔名僅為.env 的檔案, 並將檔案內容 (變數代號(key)=變數內容(value)) 轉換為系統變數
api_key = os.getenv("groq_api_key")
if not api_key:
    print("Cannot find API key")
    sys.exit(1) # 終止程式, 狀態碼 = 1 表明為異常結束狀態, 作用是系統紀錄或開發者紀錄

client = OpenAI( # Proq 相容 OpenAI 格式
    base_url="https://api.groq.com/openai/v1", # OpenAI 格式預設連到 open ai 所以要自訂 url 指向 groq
    api_key=api_key
)

In [ ]:
def call_llm(messages, parameters):
    with tracer.start_as_current_span("llm_generation") as span: # span 計篹 with: 區間時間
        span.set_attribute("model", parameters['model'])
        span.set_attribute("temperature", parameters['temperature'])
        span.set_attribute("max_tokens", parameters['max_tokens'])
        span.set_attribute("presence_penalty", parameters['presence_penalty'])
        user_input = [m['content'] for m in messages if m ['role'] == 'user'][-1]
        span.set_attribute("user_prompt", user_input)

        span.add_event("call_LLM_by_API")

        try:
            response = client.chat.completions.create(
                model=parameters['model'],
                messages=messages,
                temperature=parameters['temperature'],
                max_tokens=parameters['max_tokens'],
                presence_penalty=parameters['presence_penalty'],
                stream=False # stream=False 全部生成完再一次回傳
            )

            content = response.choices[0].message.content # 根據 "response 回傳 JSON 物件內容結構範例" 取得回復
            print(f"token: prompt = {response.usage.prompt_tokens}, completion = {response.usage.completion_tokens}, total = {response.usage.total_tokens}")
            
            span.set_attribute("prompt_token", response.usage.prompt_tokens)
            span.set_attribute("completion_token", response.usage.completion_tokens)
            span.set_attribute("total_tokens", response.usage.total_tokens)
            span.set_status(Status(StatusCode.OK))

            return content

        except Exception as e:
            span.set_status(Status(StatusCode.ERROR, str(e)))
            print(f"error: {str(e)}")
            return None

In [ ]:
# response 回傳 JSON 物件內容結構範例
# {
#   "id": "chatcmpl-123...",
#   "object": "chat.completion",
#   "created": 1712345678,
#   "model": "llama-3.3-70b-versatile",
#   "choices": [
#     {
#       "index": 0,
#       "message": {
#         "role": "assistant",
#         "content": "你好！我是 AI 助手，有什麼我可以幫你的嗎？"
#       },
#       "logprobs": null,
#       "finish_reason": "stop"
#     }
#   ],
#   "usage": {
#     "prompt_tokens": 15,
#     "completion_tokens": 25,
#     "total_tokens": 40
#   }
# }

In [ ]:
def start_chat(model = "llama-3.3-70b-versatile", temperature = 1.0, max_tokens = 512, presence_penalty = 1.1):
    parameters = {
        "model": model, # llama 3.3 on groq
        "temperature": temperature,
        "max_tokens": max_tokens,
        "presence_penalty": presence_penalty,
    }

    memory = [{"role": "system", "content": "You are a professional and helpful assistant, please respond in the user's language."}]

    print(f"{parameters["model"]} : Key in 'quit' while you want to end the chat.")
    print("=" * 100)

    while True:
        user_input = input("\n user:")

        if user_input.lower() in ['quit', 'q']:
            print("Goodbye, system off")
            break

        if not user_input.strip():
            continue

        memory.append({"role": "user", "content": user_input})
        response = call_llm(memory, parameters)

        if response:
            print(f"user: {user_input}")
            print(f"assistant: {response}")
            print("=" * 100)
            memory.append({"role": "assistant", "content": response})

In [ ]:
if __name__ == "__main__":
    start_chat()